# Лабораторная работа №2

## Дообучение ViT/ConvNeXt и извлечение признаков


### Цель

Построить воспроизводимый конвейер классификации изображений на основе предобученных ViT и ConvNeXt, сравнить их как фиксированные экстракторы признаков и исследовать эффект частичного дообучения выбранной модели при ограниченном вычислительном бюджете.

Результатом работы является не максимальная точность любой ценой, а обоснованный выбор стратегии переноса обучения с учётом качества, времени и числа обучаемых параметров.

## 1. Что используется в работе

Преподаватель предоставляет:

- размеченный датасет классификации изображений;
- фиксированное разбиение `train / validation / test`;
- подготовленное окружение с `PyTorch`, `torchvision` и `timm`;
- допустимый список компактных предобученных моделей;
- ограничение на вычислительный бюджет.

Рекомендуемые базовые модели:

- `vit_small_patch16_224`;
- `convnext_tiny`.

Студент не обучает модели с нуля. Обязательная часть включает:

1. использование обеих архитектур как фиксированных экстракторов признаков;
2. обучение одинаковой линейной головы;
3. сравнение качества, времени и ресурсоёмкости;
4. частичное дообучение только одной выбранной модели.

Ожидаемая структура проекта:

```text
lab3/
├── data/
├── outputs/
│   ├── checkpoints/
│   ├── embeddings/
│   ├── figures/
│   └── runs.jsonl
├── lab3.ipynb
└── requirements.txt
```

## 2. Краткая теоретическая справка

### 2.1. Transfer learning

Предобученная модель содержит признаки, сформированные на большом внешнем датасете. Для новой задачи используются две основные стратегии:

- **feature extraction** — backbone заморожен, обучается только новая голова;
- **fine-tuning** — часть или весь backbone адаптируется под целевой домен.

Feature extraction требует меньше памяти и времени, но ограничен качеством исходных признаков. Fine-tuning потенциально повышает качество, но увеличивает риск переобучения и стоимость эксперимента.

### 2.2. ViT и ConvNeXt

ViT разбивает изображение на патчи и моделирует их взаимодействие через self-attention. ConvNeXt сохраняет свёрточную природу, но использует решения, сближающие его с современными трансформерами.

Для лабораторной важны не только архитектурные различия, но и практический вопрос: насколько хорошо признаки каждой модели переносятся на заданный домен при одинаковом протоколе.

### 2.3. Линейный пробинг

При линейном пробинге backbone заморожен, а поверх извлечённого вектора признаков обучается линейный классификатор.
Качество линейного пробинга показывает, насколько классы уже разделимы в пространстве признаков предобученной модели.

### 2.4. Частичное дообучение

При частичном fine-tuning размораживаются только последние блоки backbone. Это компромисс между скоростью feature extraction и гибкостью полного дообучения.

Сравнение корректно только при фиксированных:

- разбиении данных;
- аугментациях;
- числе эпох;
- критерии ранней остановки;
- метриках;
- вычислительном бюджете.

## 3. Задачи

В ходе работы необходимо:

1. Проверить структуру и баланс датасета.
2. Подготовить единый pipeline загрузки и преобразований.
3. Получить embeddings от ViT и ConvNeXt при замороженных backbone.
4. Обучить одинаковые линейные классификаторы.
5. Сравнить:
   - accuracy;
   - macro F1;
   - confusion matrix;
   - время извлечения признаков;
   - время обучения;
   - объём памяти;
   - число обучаемых параметров.
6. Выбрать одну модель для частичного fine-tuning.
7. Разморозить только последний stage или последние блоки.
8. Сравнить feature extraction и partial fine-tuning.
9. Провести одно дополнительное исследование:
   - влияние разрешения входа;
   - влияние силы аугментаций;
   - влияние числа размороженных блоков;
   - влияние размера линейной головы;
   - иной согласованный фактор.
10. Сформулировать вывод о выборе архитектуры и стратегии переноса обучения.

## 4. Подготовка данных

Используйте предоставленное разбиение. Повторное случайное разбиение не допускается, так как нарушает сопоставимость результатов.

Минимальные проверки:

- число изображений по классам;
- отсутствие пересечений между split;
- корректность меток;
- размеры и форматы изображений;
- наличие дисбаланса;
- наличие почти идентичных изображений между split.

Не изменяйте тестовый набор после начала экспериментов.

In [ ]:
from pathlib import Path
import json
import time
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
import timm

DATA_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)

In [ ]:
# TODO: загрузите предоставленный manifest или сформируйте таблицу файлов.
# Обязательные столбцы:
# path, label, split

dataset_table = pd.DataFrame(columns=["path", "label", "split"])
dataset_table.head()

In [ ]:
# TODO: реализуйте проверки датасета и визуализацию распределения классов.

def validate_dataset_table(table: pd.DataFrame) -> None:
    
    pass


def plot_class_distribution(table: pd.DataFrame) -> None:
    
    pass

# validate_dataset_table(dataset_table)
# plot_class_distribution(dataset_table)

**Контрольная точка 1**

До обучения должны быть готовы:

- таблица состава датасета;
- распределение классов по split;
- проверка отсутствия пересечений;
- зафиксированный набор преобразований;
- оценка дисбаланса и выбранная стратегия его учёта.

## 5. Фиксированные экстракторы признаков

Обе модели должны использовать:

- одинаковое разрешение входа;
- одинаковые train/validation/test split;
- одинаковые базовые аугментации;
- одинаковый тип классификатора;
- одинаковый критерий выбора лучшего checkpoint.

Различие между сериями — только backbone.

In [ ]:
MODEL_NAMES = {
    "vit": "vit_small_patch16_224",
    "convnext": "convnext_tiny",
}

def build_backbone(model_name: str, num_classes: int):
    """Создать предобученную модель с новой классификационной головой."""
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes,
    )
    return model


def freeze_backbone(model: nn.Module) -> None:
    """Заморозить все параметры кроме классификационной головы."""
    
    pass


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# TODO: реализуйте Dataset и DataLoader.
# Требования:
# - train и eval transforms должны быть разделены;
# - test не использует случайные аугментации;
# - batch size выбирается с учётом памяти;
# - порядок классов фиксируется.

class ImageClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, table: pd.DataFrame, transform=None):
        
        pass

    def __len__(self):
        
        pass

    def __getitem__(self, index):
        
        pass

### Извлечение признаков

Для resource-aware варианта рекомендуется один раз извлечь embeddings замороженного backbone и сохранить их на диск. После этого линейные головы можно обучать без повторного прогона изображений через backbone.

Допускается также обучение головы напрямую поверх замороженной модели, если датасет мал и время укладывается в лимит.

In [ ]:
@torch.no_grad()
def extract_embeddings(model: nn.Module,
                       loader: DataLoader,
                       device: torch.device):
    """Вернуть embeddings, labels, пути и время извлечения."""
    
    pass


def save_embeddings(path: Path,
                    embeddings: np.ndarray,
                    labels: np.ndarray,
                    paths: list[str],
                    metadata: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        embeddings=embeddings,
        labels=labels,
        paths=np.array(paths),
        metadata=json.dumps(metadata, ensure_ascii=False),
    )

### Линейный классификатор

Для обоих backbone используется один и тот же классификатор и одинаковый протокол обучения.

In [ ]:
class LinearProbe(nn.Module):
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.classifier(x)


def train_linear_probe(
    train_embeddings: np.ndarray,
    train_labels: np.ndarray,
    val_embeddings: np.ndarray,
    val_labels: np.ndarray,
    *,
    num_classes: int,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    device: torch.device,
):
    """Обучить линейную голову и вернуть лучшую модель и историю."""
    
    pass

## 6. Экспериментальный конвейер

Каждый запуск должен фиксировать:

- backbone;
- разрешение;
- аугментации;
- seed;
- batch size;
- число эпох;
- optimizer;
- learning rate;
- weight decay;
- число обучаемых параметров;
- время извлечения признаков;
- время обучения;
- пиковую память;
- метрики validation и test;
- путь к checkpoint;
- статус выполнения.

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    name: str
    backbone: str
    strategy: str  # linear_probe / partial_finetune
    image_size: int = 224
    batch_size: int = 32
    epochs: int = 10
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    seed: int = 42
    unfrozen_blocks: int = 0


RUNS_PATH = OUTPUT_DIR / "runs.jsonl"

required_configs = [
    ExperimentConfig(
        name="vit_linear_probe",
        backbone=MODEL_NAMES["vit"],
        strategy="linear_probe",
    ),
    ExperimentConfig(
        name="convnext_linear_probe",
        backbone=MODEL_NAMES["convnext"],
        strategy="linear_probe",
    ),
]

required_configs

In [ ]:
def set_global_seed(seed: int) -> None:
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def run_experiment(config: ExperimentConfig,
                   dataset_table: pd.DataFrame) -> dict:
    """Выполнить один воспроизводимый эксперимент."""
    
    pass

**Контрольная точка 2**

Конвейер считается готовым, если:

- linear probe для ViT и ConvNeXt запускается одним интерфейсом;
- embeddings сохраняются и повторно используются;
- журнал содержит все существенные параметры;
- лучший checkpoint выбирается по validation, а не test;
- повторный запуск не дублирует завершённые серии;
- ошибки одной серии не уничтожают результаты остальных.

## 7. Частичное дообучение и дополнительное исследование

После сравнения линейных проб выберите одну модель для partial fine-tuning.

Выбор должен опираться минимум на:

- качество validation;
- время;
- число параметров;
- размер embeddings или checkpoint;
- устойчивость по классам.

Разморозьте только последний stage или несколько последних блоков. Полное дообучение всей модели в обязательную часть не входит.

In [ ]:
# TODO: обоснуйте выбор модели.

selected_backbone = ""
selection_rationale = ""


def unfreeze_last_blocks(model: nn.Module,
                         backbone_name: str,
                         num_blocks: int) -> None:
    """Разморозить последние блоки выбранного backbone."""
    raise NotImplementedError


partial_finetune_config = ExperimentConfig(
    name="selected_partial_finetune",
    backbone=selected_backbone or MODEL_NAMES["convnext"],
    strategy="partial_finetune",
    epochs=5,
    learning_rate=1e-5,
    unfrozen_blocks=1,
)

partial_finetune_config

### Дополнительное исследование

Выберите один фактор и сформулируйте гипотезу:

- разрешение входа;
- сила аугментаций;
- число размороженных блоков;
- размер и структура головы;
- иной согласованный фактор.

Дополнительная серия не должна превышать **2–3 конфигурации**. Это ограничение введено для сохранения четырёхчасового бюджета.

In [ ]:
research_question = ""
hypothesis = ""
extra_configs = []

# TODO: добавьте 2–3 сопоставимые конфигурации.

## 8. Оценка, анализ и сдача

Обязательные метрики:

- accuracy;
- macro F1;
- per-class recall;
- confusion matrix;
- время извлечения признаков;
- время обучения;
- число обучаемых параметров;
- пиковая GPU-память или, при CPU, максимальное потребление RAM.

Дополнительно проанализируйте:

- классы, которые разделяются уже линейно;
- классы, для которых требуется адаптация признаков;
- случаи деградации после partial fine-tuning;
- признаки переобучения;
- соотношение прироста качества и вычислительной стоимости.

In [ ]:
# TODO: выполнить обязательные эксперименты и собрать таблицу результатов.

results = pd.DataFrame()
results

In [ ]:
# TODO: сформировать сводную таблицу.
# Минимальные поля:
# name, backbone, strategy, accuracy, macro_f1,
# trainable_params, feature_time, train_time,
# peak_memory_mb, checkpoint_size_mb.

def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    
    pass

In [ ]:
# TODO: построить:
# 1. quality vs training time;
# 2. quality vs trainable parameters;
# 3. confusion matrix для лучшей linear-probe и partial-finetune моделей.
#
# Каждый график должен быть отдельной фигурой.

### Обязательные артефакты

1. Анализ и проверка датасета.
2. Зафиксированные transforms и split.
3. Реализация Dataset/DataLoader.
4. Реализация заморозки backbone.
5. Embeddings ViT и ConvNeXt.
6. Две linear-probe серии.
7. Сохранённые checkpoints и журнал `runs.jsonl`.
8. Обоснованный выбор модели для partial fine-tuning.
9. Одна partial fine-tuning серия.
10. Одно дополнительное исследование.
11. Сводная таблица качества и ресурсоёмкости.
12. Не менее трёх графиков.
13. Confusion matrix для ключевых моделей.
14. Анализ ошибок и итоговый вывод.

### Итоговый вывод

В выводе необходимо ответить:

- какой backbone дал более переносимые признаки;
- оправдано ли partial fine-tuning;
- какие классы выиграли и проиграли;
- насколько прирост качества соответствует дополнительным затратам;
- какую стратегию следует выбрать при аналогичном вычислительном бюджете.


## Критерии оценивания

- Подготовка и анализ данных
- Реализация общего pipeline 
- ViT linear probe 
- ConvNeXt linear probe 
- Сравнение экстракторов 
- Partial fine-tuning 
- Дополнительное исследование 
- Представление результатов 
- Анализ и выводы

### Условия зачёта

Работа не засчитывается, если:

- модели сравнивались на разных split или transforms без обоснования;
- test использовался для настройки;
- приведён только лучший результат без полного журнала;
- выводы не подтверждаются сохранёнными экспериментами.